# Fractum Distributed Encodes — Kaggle Worker

Runs a [DistributedEncodes](https://github.com/FractumSeraph/DistributedEncodes) worker on a free Kaggle CPU session.

## Quick start

1. **Settings** (sidebar): *Internet* → **On**, *Accelerator* → **None** — the encoder (SVT-AV1) is CPU-only; a GPU adds nothing and burns your weekly GPU quota.
2. Edit the **CONFIG** cell below (username, worker name, jobs).
3. **Save Version → Save & Run All (Commit)** — the worker runs headless for ~11.5 h and shuts itself down gracefully before Kaggle's 12 h hard kill. You can close the browser.
   (Interactive sessions are killed ~40 min after the tab closes, so always use *Save & Run All* for long runs.)
4. Re-run the notebook — or schedule it (*Notebook → Schedule a notebook run*) — to keep contributing.

## What this notebook does differently

* **No `apt-get` / manual FFmpeg installs.** Apt's FFmpeg 4.4 and johnvansickle's 7.0.x static build both lack `libsvtav1` ≥ 7.1, so the worker rejects them and downloads its own BtbN static build anyway. The old setup spent ~1 min installing FFmpeg twice for nothing.
* **`--no-tui`.** The Textual TUI in a notebook is just megabytes of ANSI escape codes in the saved output (and can push Kaggle's log past its size limit). Plain logs instead, with `\r`-style progress lines sampled once a minute.
* **`textual` + `requests` pre-installed** so the worker's auto-installer doesn't pip-install and re-exec itself mid-start.
* **Temp files in `/kaggle/tmp`**, not `/kaggle/working` — working/ is capped at ~20 GB and snapshotted into the notebook output on save, which makes commits slow.
* **Time-boxed supervisor**: graceful shutdown before the 12 h session limit (so the last chunk isn't lost to a hard kill), crash restarts with backoff, and no fresh launches in the final minutes.


In [ ]:
# ============================== CONFIG ==============================
USERNAME   = "FractumSeraph"  # Scoreboard name
WORKERNAME = "Kaggle"         # Identifier for this machine
JOBS       = 2                # Parallel encode threads. 2 keeps the CPU busy
                              # while the other job is downloading/uploading.

MAX_RUNTIME_HOURS = 11.5      # Kaggle batch sessions are hard-killed at 12 h —
                              # stop gracefully before that.
NO_RELAUNCH_FINAL_MINUTES = 20  # Don't launch a fresh worker this close to the
                                # deadline; a new job would likely be cut off.
STATUS_EVERY_SECS = 60        # How often to echo the workers' progress line
                              # (keeps 12 h of logs small instead of 2 lines/sec).

EXTRA_ARGS = []               # e.g. ["--series-id", "42"], ["--wallet", "ADDR"],
                              #      ["--daily-quota", "50"], ["--max-size-mb", "4000"]

# Optional: store WORKER_SECRET under "Add-ons -> Secrets" instead of
# hardcoding it. Falls back to the worker's built-in default if absent.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WORKER_SECRET"] = UserSecretsClient().get_secret("WORKER_SECRET")
    print("[*] WORKER_SECRET loaded from Kaggle secrets.")
except Exception:
    pass  # no secret configured — worker uses its public default


In [ ]:
# ============================== SETUP ==============================
# Deliberately absent (dead weight in the old notebook):
#   * apt-get install ffmpeg      — apt's 4.4 lacks libsvtav1 >= 7.1; rejected.
#   * johnvansickle static ffmpeg — 7.0.x, also rejected. worker_template.py
#     downloads its own BtbN static build with libsvtav1 on first run.
import os, subprocess, sys

REPO_URL = "https://github.com/FractumSeraph/DistributedEncodes.git"
BRANCH   = "main"

# Keep the repo + multi-GB encode temp files OUT of /kaggle/working:
# it's capped at ~20 GB and snapshotted into the notebook output on save.
BASE_DIR = "/kaggle/tmp" if os.path.isdir("/kaggle") else os.getcwd()
os.makedirs(BASE_DIR, exist_ok=True)
REPO_DIR = os.path.join(BASE_DIR, "DistributedEncodes")

print("--- Installing Python deps ---")
# Pre-installing textual skips the worker's own pip-install + self-restart.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "requests", "textual>=0.20"], check=True)

print("--- Cloning / updating repo ---")
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1",
                    "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard",
                    f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO_URL, REPO_DIR], check=True)
print(f"Repo ready at {REPO_DIR}")


In [ ]:
# ============================== RUN ==============================
# Supervisor for worker_template.py, tuned for Kaggle:
#   * --no-tui       : plain logs instead of megabytes of ANSI escape codes.
#   * output pump    : \r-overwritten status/progress lines are sampled once
#                      per STATUS_EVERY_SECS so 12 h of logs stays tiny.
#   * deadline       : graceful stop (SIGINT + pause-menu "s") before Kaggle
#                      hard-kills the session at 12 h.
#   * crash restarts : exponential backoff; no fresh launches near the deadline.
import re, signal, subprocess, threading, time

ANSI_RE  = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")
DEADLINE = time.monotonic() + MAX_RUNTIME_HOURS * 3600
started_at = time.time()

def remaining():
    return DEADLINE - time.monotonic()

def _pump(stream):
    """Forward worker output. Normal lines pass through; \r-overwritten
    status lines (which never get a newline) are sampled periodically."""
    buf, last_status = b"", 0.0
    while True:
        chunk = stream.read1(4096)
        if not chunk:
            tail = ANSI_RE.sub("", buf.split(b"\r")[-1].decode("utf-8", "replace")).strip()
            if tail:
                print(tail, flush=True)
            return
        buf += chunk
        while b"\n" in buf:
            line, buf = buf.split(b"\n", 1)
            text = ANSI_RE.sub("", line.split(b"\r")[-1].decode("utf-8", "replace")).rstrip()
            if text:
                print(text, flush=True)
        if b"\r" in buf:  # unfinished status line — keep only the newest segment
            *done, buf = buf.split(b"\r")
            now = time.monotonic()
            if now - last_status >= STATUS_EVERY_SECS:
                for seg in reversed(done):
                    text = ANSI_RE.sub("", seg.decode("utf-8", "replace")).strip()
                    if text:
                        print(f"[status] {text}", flush=True)
                        last_status = now
                        break

def launch():
    cmd = [sys.executable, "-u", "worker_template.py",
           "--username", USERNAME, "--workername", WORKERNAME,
           "--jobs", str(JOBS), "--no-tui", *EXTRA_ARGS]
    env = dict(os.environ, PYTHONUNBUFFERED="1", COLUMNS="200")
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env,
                            stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT)
    t = threading.Thread(target=_pump, args=(proc.stdout,), daemon=True)
    t.start()
    return proc, t

def stop_gracefully(proc, pump_t, grace=120):
    """SIGINT opens the worker's pause menu on stdin; "s" = clean abort
    (kills ffmpeg children, exits 0). Escalate if it doesn't comply."""
    if proc.poll() is not None:
        return
    try:
        proc.send_signal(signal.SIGINT)
        time.sleep(2)  # let the pause menu come up
        proc.stdin.write(b"s\n")
        proc.stdin.flush()
    except Exception:
        pass
    try:
        proc.wait(timeout=grace)
    except subprocess.TimeoutExpired:
        print("[!] Worker didn't stop in time — killing.", flush=True)
        proc.kill()
        proc.wait()
    pump_t.join(timeout=10)

backoff, runs = 10, 0
print(f"[*] Worker window: {MAX_RUNTIME_HOURS} h from now.")
try:
    while remaining() > NO_RELAUNCH_FINAL_MINUTES * 60:
        runs += 1
        run_started = time.monotonic()
        print(f"[*] Launching worker (run #{runs}, {remaining()/3600:.1f} h left)...")
        proc, pump_t = launch()
        while proc.poll() is None and remaining() > 0:
            time.sleep(5)
        if proc.poll() is None:  # deadline hit while still running
            print("[*] Session time limit reached — stopping worker gracefully...")
            stop_gracefully(proc, pump_t)
            break
        pump_t.join(timeout=10)
        healthy = (time.monotonic() - run_started) > 600
        backoff = 10 if healthy else min(backoff * 2, 300)
        print(f"[!] Worker exited with code {proc.returncode}. "
              f"Restarting in {backoff} s...")
        time.sleep(min(backoff, max(remaining(), 0)))
except KeyboardInterrupt:
    print("[*] Interrupted — stopping worker gracefully...")
    try:
        stop_gracefully(proc, pump_t)
    except NameError:
        pass

print(f"[*] Done. {(time.time() - started_at) / 3600:.2f} h wall time, "
      f"{runs} worker run(s).")
